In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-5: Case study: training a reasoning model with RLVR

## Where this session fits

> **The limitation we start from: RLLM-4 left two independent choices open: the reward source, and a baseline without a critic. RLVR with GRPO makes both.**

- **The question:** what does one recipe, with a verifiable reward (RLVR) and a critic-free group baseline, do to a model on long reasoning chains?
- **Position: session 5 of the RL for LLMs series (RLLM-1 to RLLM-14)**
    - RLLM-4 showed that PPO needs a critic, which costs a network and is fragile on long completions
    - **RLVR** (reinforcement learning with verifiable rewards): a reward that a program checks
    - **GRPO**: the optimizer (Intro session C2)
    - this session runs RLVR with plain GRPO
    - next: RLLM-6, GRPO and its refinements
- **Reward sources (RLLM-1 map)**
    - people compare (RLHF) (RLLM-2)
    - an AI compares (RLAIF) (RLLM-3)
    - **a program checks (RLVR) (RLLM-5)**
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what a reasoning model does differently, and what the paper reports about RL on a **base model** (the model before post-training)
2. say why RLVR and GRPO are two independent choices, which this recipe makes together, and what each replaces
3. compute a rule-based reward and its group advantage for a group of completions
4. tell an outcome reward from a process reward, and say what each does for credit assignment
5. read what R1-Zero reported, and the stages of the DeepSeek-R1 recipe
6. read the signals of an RLVR run with GRPO, and name what the training logs cannot show


# The case study: road map

**Goal:** train a model to reason with RLVR: a reward that a program checks rather than a learned one.

> **The idea in one line: reward correct answers by rule (RLVR), let GRPO raise what earns them, and see what a published recipe reported.**

1. **The behavior:** what a reasoning model does differently, with a toy example
2. **The reward source (RLVR):** a sum of rule-based checks, and why a program was chosen over a learned model
3. **The optimizer (GRPO):** why GRPO suits it, and a worked group with rewards turned into advantages
4. **Outcome and process rewards:** what the reward sees of a completion, and what it misses
5. **The results:** what the DeepSeek-R1 paper reported for RL straight on a base model
6. **The recipe:** why a short SFT step comes first, and the stages of DeepSeek-R1
7. **Reading a run:** the signals of an RLVR run, and what the logs cannot show


# Step 1 of 7: What a reasoning model does

**Why this section?** The case study trains one behavior: thinking before answering.

> **A reasoning model writes a long chain of thought, with reflection and revision, before it gives its answer.**

- the **chain of thought** is step-by-step reasoning in the completion
- **reflection:** looking back at the work so far, and judging the solution and the strategy
- **revision:** changing the current approach
- the completion has two marked parts, so a program can find the answer
    - the reasoning, between `<think>` and `</think>`
    - the answer, between `<answer>` and `</answer>`
    - the markers also let the user hide the reasoning
- the paper reports that RL alone on a base model raised this behavior (Step 5)
    - our reading: the base model did not show it at first

**A toy example** (real tasks are much harder; this one needs no long reasoning)

- prompt: "Find a number that, multiplied by 3 and then increased by 7, equals 22."
- a plain completion: "The answer is 8" (a guess, no work shown, wrong)
    - illustrative only: a base model may well compute this directly
- a reasoning completion, with a failed first attempt
    - `<think>` try x = 8: 3 times 8 is 24, plus 7 is 31
    - reflection: "that does not match 22, so the guess is wrong"
    - revision: "instead solve 3x + 7 = 22: subtract 7 to get 3x = 15, so x = 5"
    - check: 3 times 5 is 15, and 15 plus 7 is 22
    - `</think>` then `<answer>5</answer>`


## Step 2 of 7: The reward source, RLVR: rules, not a learned model

**Why this section?** To raise this behavior with RL we need a reward. RLLM-2 and RLLM-3 built one from comparisons. Here a program computes it.

> **RLVR (reinforcement learning with verifiable rewards) means the reward is rule-based: a sum of checks on the completion, computed by a program with nothing to train.**

- an **accuracy** reward: is the final answer correct?
    - a **verifiable** question (hence the name RLVR): we may not know how to find the answer, but given it we can check it
    - for example, a number in a math problem; code is checked with tests
- a **format** reward: are the `<think>` and `<answer>` markers present and in order?
    - without them the answer cannot be found in the completion
    - **keep this:** a completion cut off before `</answer>` fails both checks 
        - RLLM-6 returns to this: a length limit can cut completions off
- other terms are possible: a reward for staying in one language (used by R1, see the recipe)

**Why not a learned reward model (RLLM-2)?**

> **Rules are not over-optimized by the same mechanism as a learned proxy, but they can still be gamed: reward hacking (RLLM-2) happens with rules too.**

- the DeepSeek-R1 paper reports that a neural reward model may suffer from reward hacking in large-scale RL
- our inference: a program has no learned parameters to drift, and needs no labeler for each completion
- the price: an answer key for each prompt, and a check only of what its author wrote
- our illustration: the format reward is earned by empty markers around nonsense, so the accuracy check must carry the meaning


## Step 3 of 7: The optimizer, GRPO, and a worked group

**Why this section?** The reward source (RLVR) and the optimizer are separate choices. We say why GRPO suits RLVR, then compute one group.

> **RLVR and GRPO are independent: PPO could use a program reward too. GRPO suits it: no critic, and many samples per prompt are cheap to score.**

- RLLM-4's two choices, both made here
    - **RLVR, the reward source:** no reward model to train or to drift
    - **GRPO, the optimizer:** a group baseline instead of a critic
        - the paper's reason: it drops the critic, a network as large as the policy
    - the paper trains R1-Zero with this pair; either choice could be swapped
- GRPO samples a group of $G$ completions per prompt and compares them (Intro session C2)
    - our reading: scoring a completion costs one program call, so a large group is cheap (the paper does not give this reason)
- GRPO's loss also carries a KL term to a reference model (Intro session C2): RLLM-6 asks whether to keep it

**A worked group: rewards and advantages**

> **Rewards are centered and scaled within the group, so a completion is judged against its siblings, not against a fixed bar.**

- one prompt, $G = 5$ completions: accuracy and format both count 1
    - illustrative values, not the paper's
| completion | what happened | accuracy | format | total reward | advantage |
|:--|:--|:--|:--|:--|:--|
| 1 | correct, markers present | 1 | 1 | 2 | +0.73 |
| 2 | correct, markers present | 1 | 1 | 2 | +0.73 |
| 3 | wrong answer, markers present | 0 | 1 | 1 | -1.10 |
| 4 | correct, but no think markers | 1 | 0 | 1 | -1.10 |
| 5 | correct, markers present | 1 | 1 | 2 | +0.73 |

- the mean of the total rewards is $1.60$ and their standard deviation is $0.55$ (divisor $G - 1$, as in C2)
- the advantage is $(\text{total reward} - \text{mean}) / \text{std}$, and every token of the completion gets it
- implementations differ: with divisor $G$ the advantages are $+0.82$ and $-1.22$ instead of $+0.73$ and $-1.10$

**Reading the table**

> **Keep this: if every completion in a group earns the same total, every advantage is $0$ and nothing is learned. RLLM-6 shows a method that filters such groups out.**
<!-- seed-plant: all-equal-group -->

- completions 3 and 4 have total reward 1, below the mean 1.6, so both are pushed down
    - completion 3 has the wrong answer
    - completion 4 is correct but lacks the markers, which cost a point
- completions 1, 2 and 5 are pushed up equally: the rule cannot tell them apart
- completions 3 and 4 are penalized equally, though one is wrong and one only lacks markers
- all totals are non-negative, yet two advantages are negative: centering makes the comparison relative


## Step 4 of 7: Outcome rewards and process rewards

**Why this section?** RLLM-4 gave the critic two roles: baseline, and credit assignment (which token earned what). The group just computed replaces the first. What of the second?

> **An outcome reward scores the whole completion. A process reward scores each step, so only it shows where reasoning went wrong.**

- an **outcome reward** gives one reward at the end: right or wrong
    - it can come from a program or from a learned outcome reward model (ORM)
    - what the rule-based reward of this session does
- a **process reward model** (PRM) gives a reward for each reasoning step
    - for example, step 3 of a solution is wrong: only that step is marked down
    - it needs a label for every step, which is costly
- the DeepSeek-R1 paper reports not using a PRM for its reasoning RL
    - it lists reward hacking of a learned PRM among its reasons

**What this means for credit assignment**

<!-- seed-pay: critic-two-roles -->

> **From RLLM-4: the critic's second role is credit assignment. GRPO's one advantage per completion gives it up, so every token shares the outcome.**

- with an outcome reward, a correct answer lifts every token of the chain, including a wasted step (our reading)
- a PRM would restore some credit assignment, at the cost of step labels and a new proxy to hack
- the recipe accepts the coarse signal
    - RLLM-6 shows weak spots in GRPO's objective and names credit assignment as still open


## Step 5 of 7: What R1-Zero reported

**Why this section?** Did the recipe work? The paper reports one RL run straight on a base model, with rule-based rewards.

> **The revised paper (arXiv v2) reports that R1-Zero's AIME 2024 pass@1 (the share of problems one sample solves) rose from 15.6% to 77.9% during RL, and 86.7% with majority voting (taking the most common answer of many samples).**

<img src="images/rllm5_results.png" width=56%>

- **R1-Zero:** RL on DeepSeek-V3-Base with accuracy and format rewards, no SFT first
- **AIME 2024:** competition math with exact numeric answers
- **pass@1:** the share of problems that one sample solves
- **majority voting (self-consistency):** sample many completions, take the most common answer: a different measure from pass@1
    - the first arXiv version reported 71.0% pass@1; we cite the revised one

**What else the paper reports about R1-Zero, and how far to read it**

- the paper also reports completion length growing steadily, and calls a jump in reflection an "aha moment" (its framing)
    - our reading: part of the length growth may be a loss artifact
    - later work finds reflection already in base models (our reading)
    - RLLM-6 returns to the length part
- **but** it also reports poor readability and language mixing (English and Chinese in one chain of thought)
- RLLM-6 uses other runs, not these numbers


## Step 6 of 7: From R1-Zero to R1: the cold start and the recipe

**Why this section?** R1-Zero reasons well but is not a finished assistant. The paper's answer is a pipeline.

> **The DeepSeek-R1 paper's stated reason for a short SFT step first: R1-Zero produced completions that were hard to read and mixed languages.**

- **cold-start SFT:** a short SFT step on examples in the new style
    - it gives a readable start (a human-friendly style)
    - it does not guarantee correct reasoning (our reading)
- R1-Zero was strong (Step 5) but hard to read; the stages below address that
- our reading, a general second reason: RL raises only what the model already does sometimes (C1)
    - a weak base model may need SFT first
    - V3's 15.6% start suggests that was not the issue here

**The DeepSeek-R1 recipe, stage by stage**

> **Reasoning RL is the middle of a pipeline: SFT before it, more SFT and a second RL stage after, then distillation.**

<img src="images/rllm5_r1.png" width=85%>

- **reasoning RL:** GRPO with accuracy and format rewards, plus a language-consistency reward against language mixing
- **rejection sampling and SFT:** sample many completions, keep those that pass the check, then SFT on the kept ones
    - **rejection sampling:** generate many, keep only the accepted
        - RLLM-7 to RLLM-14 (Part 2, alternatives to GRPO) build on it
- **RL for helpfulness and harmlessness:** learned reward models for general prompts, rules for reasoning prompts
    - this stage uses a learned reward model: real recipes mix reward sources (the map of RLLM-1)
- **distillation:** smaller models are fine-tuned on its completions: imitation (RLLM-1) of a model


## Step 7 of 7: Reading a run, what to watch

**Why this section?** A run fails quietly. The training logs show whether the optimizer is working, but not whether the model is getting better.

> **With RLVR the training reward can rise while real ability does not. Only held-out gold or human evaluation shows that: the training logs cannot.**

<img src="images/rllm4_signals.png" width=92%>

- **synthetic illustration** (hand-made curves, not a real run): the figure shows 3 of the signals of the next slide
    - orange is what the run sees
    - green needs held-out evaluation, so it is not in the logs
- a rule can be gamed too (Step 2), so a rising reward alone proves little

**The signals of an RLVR run with GRPO, and what a warning looks like**

| Signal | What it measures | A warning looks like (rules of thumb, our reading) |
|:--|:--|:--|
| mean reward | what the policy optimizes | rises while held-out results do not |
| completion length | what is being produced | grows: the reward may favor length |
| all-equal groups | share of prompts whose completions all got the same reward | high: their advantages are all $0$, so no learning signal |
| truncation | share of completions cut off at the length limit | high: the reward judges unfinished answers |
| KL to the reference, entropy, clip fraction | how far the policy moved (KL), how spread the next-token distribution is (entropy, 0 = one token), how many tokens hit the clip | KL grows without bound or stays at $0$. Entropy collapses early. Clip fraction is informative only if a batch is reused for several updates (with one update per batch the ratio is 1 and it is $0$) |

- **a healthy run (our reading):** reward rises slowly, length stays stable or grows, entropy falls gently, few all-equal groups, few truncations
- no value loss: GRPO has no critic, so all-equal groups and truncations are counts read off the batch (Step 3 showed an all-equal group)
- **not in the logs:** held-out gold or human evaluation, the signal that says whether the model improved


# What we established, and what it sets up

- **Established**
    - a reasoning model thinks in a long chain of thought, with reflection and revision
    - the reward is a program: accuracy and format checks, no model to train, but rules can be gamed
    - GRPO fits it: no critic, and rewards centered and scaled within the group (C2) give advantages
    - an outcome reward is coarse: a process reward gives credit per step, but needs step labels
    - the paper reports R1-Zero rising from 15.6% to 77.9% on AIME 2024, with poor readability and language mixing
    - RLVR is the reward source, GRPO the optimizer: independent choices that this recipe makes together
    - a run is read from its training signals, but only held-out evaluation shows real improvement
    - the R1 recipe: cold-start SFT, reasoning RL, rejection sampling and SFT, RL for helpfulness and harmlessness, distillation
- **Sets up**
    - **the limits:** rules need verifiable tasks and an answer key
    - R1 used plain GRPO and reports results: later papers report that details of the objective decide whether it works
    - next session, RLLM-6: GRPO and its refinements
